In [1]:
# Install the original QLoRA stack and the shared RAG evaluation packages.
!pip install -q -U "transformers>=4.48,<5" "datasets>=3.0" "accelerate>=1.0" "peft>=0.14" "trl>=0.24,<1" "bitsandbytes>=0.45" "huggingface_hub>=0.27" "pandas>=2.0" "tqdm>=4.66" rapidfuzz sacrebleu bert-score==0.3.13


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Import the original fine-tuning stack and set reproducibility.
import json, random, re, unicodedata, numpy as np, pandas as pd, torch
from pathlib import Path
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score
SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this QLoRA notebook.")
print(torch.cuda.get_device_name(0))


Tesla T4


In [3]:
# Log in to Hugging Face before loading the gated Llama model.
from huggingface_hub import notebook_login
notebook_login()


In [4]:
# Keep the original fine-tuning configuration and requested GitHub data paths.
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
TRAIN_URL = "https://raw.githubusercontent.com/RamijWasithRahat/Bangla-Agriculture-Chatbot/main/Data/Bangla_Agriculture_QA_Train_800.json"
TEST_URL = "https://raw.githubusercontent.com/RamijWasithRahat/Bangla-Agriculture-Chatbot/main/Data/Bangla_Agriculture_QA_Test_200.json"
OUTPUT_DIR = "./llama-3.2-1b-bangla-agriculture-qlora"
ADAPTER_DIR = "./llama-3.2-1b-bangla-agriculture-adapter"
RESULT_DIR = Path("./finetuned_results"); RESULT_DIR.mkdir(exist_ok=True)
MAX_LENGTH, NUM_EPOCHS, TRAIN_BATCH_SIZE = 512, 5, 4
GRAD_ACCUM_STEPS, LEARNING_RATE = 4, 2e-4
GENERATION_BATCH_SIZE, MAX_NEW_TOKENS = 8, 320
SYSTEM_PROMPT = "তুমি বাংলাদেশের কৃষি বিষয়ক প্রশ্নের উত্তর দেওয়ার জন্য একটি সহায়ক সহকারী। শুধু প্রশ্নের প্রাসঙ্গিক উত্তর বাংলায় দাও। যেখানে সংখ্যা, সময়, দূরত্ব, পরিমাণ বা নির্দিষ্ট তথ্য আছে সেখানে তা সঠিকভাবে উল্লেখ করো। অপ্রয়োজনীয় তথ্য তৈরি করো না।"


In [5]:
# Load all 800 training examples and the untouched 200-example test set.
train_raw = load_dataset("json", data_files=TRAIN_URL, field="qa_pairs", split="train")
test_raw = load_dataset("json", data_files=TEST_URL, field="qa_pairs", split="train")
assert len(train_raw) == 800 and len(test_raw) == 200
print(train_raw, test_raw)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'question', 'reference_answer'],
    num_rows: 800
}) Dataset({
    features: ['id', 'question', 'reference_answer'],
    num_rows: 200
})


In [6]:
# Convert the original QA records into prompt-completion training format.
def to_sft_format(example):
    return {
        "prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": example["question"].strip()}],
        "completion": [{"role": "assistant", "content": example["reference_answer"].strip()}],
    }
train_sft = train_raw.map(to_sft_format, remove_columns=train_raw.column_names)
print(train_sft[0])


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

{'prompt': [{'content': 'তুমি বাংলাদেশের কৃষি বিষয়ক প্রশ্নের উত্তর দেওয়ার জন্য একটি সহায়ক সহকারী। শুধু প্রশ্নের প্রাসঙ্গিক উত্তর বাংলায় দাও। যেখানে সংখ্যা, সময়, দূরত্ব, পরিমাণ বা নির্দিষ্ট তথ্য আছে সেখানে তা সঠিকভাবে উল্লেখ করো। অপ্রয়োজনীয় তথ্য তৈরি করো না।', 'role': 'system'}, {'content': 'আলু কীভাবে সংরক্ষণ করতে হবে?', 'role': 'user'}], 'completion': [{'content': 'অস্থায়ী শেড থেকে বাছাই শেষে বিক্রির জন্য বিক্রয় করা অথবা সংরক্ষণের জন্য কোল্ড ষ্টোরে রাখা যাবে।', 'role': 'assistant'}]}


In [7]:
# Load the tokenizer with the original right-padding training setup.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(tokenizer.pad_token, tokenizer.eos_token)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

<|eot_id|> <|eot_id|>


In [8]:
# Check sequence lengths without changing the original maximum length.
def formatted_length(example):
    text = tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False, add_generation_prompt=False)
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])
lengths = [formatted_length(x) for x in train_sft]
print("max:", max(lengths), "over limit:", sum(x > MAX_LENGTH for x in lengths))


max: 661 over limit: 233


In [9]:
# Configure the original 4-bit NF4 QLoRA quantization.
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
print(compute_dtype)


torch.bfloat16


In [10]:
# Load and prepare the original base model for k-bit training.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto", dtype=compute_dtype)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print("4-bit:", getattr(model, "is_loaded_in_4bit", False))


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

4-bit: True


In [11]:
# Keep the original LoRA adapter configuration unchanged.
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
print(peft_config)


LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'up_proj', 'v_proj', 'k_proj', 'down_proj', 'q_proj', 'o_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_ty

In [12]:
# Keep the original five-epoch SFT training arguments unchanged.
training_args = SFTConfig(
    output_dir=OUTPUT_DIR, num_train_epochs=NUM_EPOCHS, per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS, learning_rate=LEARNING_RATE, weight_decay=0.01,
    warmup_ratio=0.05, lr_scheduler_type="cosine", max_grad_norm=0.3, max_length=MAX_LENGTH,
    completion_only_loss=True, packing=False, gradient_checkpointing=True, bf16=use_bf16,
    fp16=not use_bf16, optim="paged_adamw_8bit", save_strategy="epoch", save_total_limit=2,
    logging_steps=10, seed=SEED, data_seed=SEED, report_to="none",
)
print(training_args)


SFTConfig(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=True,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eos_token=<

In [13]:
# Create the original SFTTrainer with the LoRA adapter.
trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_sft, processing_class=tokenizer, peft_config=peft_config)
trainer.model.print_trainable_parameters()


Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [14]:
# Fine-tune Llama 3.2 1B on the 800 agriculture examples.
train_result = trainer.train()
print(train_result.metrics)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
10,1.101200
20,0.926200
30,0.802500
40,0.737700
50,0.679300
60,0.558400
70,0.522800
80,0.491800
90,0.464400
100,0.405800


{'train_runtime': 7730.7858, 'train_samples_per_second': 0.517, 'train_steps_per_second': 0.032, 'total_flos': 1.1977834898128896e+16, 'train_loss': 0.3508322377204895}


In [15]:
# Save the trained LoRA adapter and tokenizer.
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved:", ADAPTER_DIR)


Saved: ./llama-3.2-1b-bangla-agriculture-adapter


In [16]:
# Restore the original deterministic generation setup.
trainer.model.config.use_cache = True
trainer.model.eval()
terminators = [tokenizer.eos_token_id]
eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
if isinstance(eot_id, int) and eot_id >= 0 and eot_id not in terminators:
    terminators.append(eot_id)
def make_generation_prompt(question):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question.strip()}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [17]:
# Generate one test answer with the original decoding settings.
def generate_one(question, max_new_tokens=MAX_NEW_TOKENS):
    old_side = tokenizer.padding_side; tokenizer.padding_side = "left"
    inputs = tokenizer(make_generation_prompt(question), return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.inference_mode():
        output = trainer.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=terminators, pad_token_id=tokenizer.pad_token_id)
    generated = output[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
    tokenizer.padding_side = old_side
    return answer
print(generate_one(test_raw[0]["question"]))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


গাছের গোড়া সবুজ, গাঢ় সবুজ।


In [18]:
# Generate all 200 fine-tuned predictions and track max-length truncation.
from tqdm.auto import tqdm
def generate_batch(questions, max_new_tokens=MAX_NEW_TOKENS):
    old_side = tokenizer.padding_side; tokenizer.padding_side = "left"
    prompts = [make_generation_prompt(q) for q in questions]
    batch = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    width = batch["input_ids"].shape[1]
    with torch.inference_mode():
        outputs = trainer.model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=terminators, pad_token_id=tokenizer.pad_token_id)
    answers, flags = [], []
    for output in outputs:
        generated = output[width:]
        ids = generated.tolist()
        answers.append(tokenizer.decode(generated, skip_special_tokens=True).strip())
        flags.append(len(ids) >= max_new_tokens and not any(t in terminators for t in ids))
    tokenizer.padding_side = old_side
    return answers, flags
all_predictions, all_truncated = [], []
for start in tqdm(range(0, len(test_raw), GENERATION_BATCH_SIZE)):
    preds, flags = generate_batch(test_raw[start:start+GENERATION_BATCH_SIZE]["question"])
    all_predictions.extend(preds); all_truncated.extend(flags)
assert len(all_predictions) == len(test_raw) == 200


  0%|          | 0/25 [00:00<?, ?it/s]

In [19]:
# Build the fine-tuned prediction table.
predictions_df = pd.DataFrame({
    "id": test_raw["id"],
    "question": test_raw["question"],
    "gold": test_raw["reference_answer"],
    "prediction": all_predictions,
    "truncated": all_truncated,
})
display(predictions_df.head())


,id,question,gold,prediction,truncated
0,801,চুইঝাল সম্পর্কে গাছের গোড়া বিষয়ে কী বলা হয়েছে?,সার দেয়ার সময় গাছের গোড়ায় মাটি হালকা করে ক...,"গাছের গোড়া সবুজ, গাঢ় সবুজ।",False
1,802,পানিকচুর বন্যাপ্রবণ জায়গায় চারা লাগানোর সময় কী?,যে সব জায়গা বন্যার পানিতে তলিয়ে যাবার সম্ভাব...,চারা রোপণের এক মাস পর আগাম মাসে চারা রোপণ করতে...,False
2,803,সুপারি: বীজতলার মাটিতে বালুর পরিমাণ প্রসঙ্গে ক...,বীজতলার মাটিতে বালুর পরিমাণ কম থাকলে কিছু ভিটি...,"বীজতলার মাটিতে বালুর পরিমাণ প্রশাখাশীল, নিচু ও...",False
3,804,মুখীকচু সম্পর্কে ডাবল সারি পদ্ধতির রোপণ দূরত্ব...,ডাবল সারি পদ্ধতিতে রোপণ দূরত্ব: ৭৫ সেমি × ৬০ স...,ডাবল সারি পদ্ধতিতে রোপণ দূরত্ব ২৫ সে.মি. (মাটি...,False
4,805,কফির মাটি ও জমি নির্বাচন সম্পর্কে কী বলা হয়েছে?,বাণিজ্যিকভাবে কফি চাষ করে অধিক ফলনের জন্য কমপক...,কফি উৎপাদন প্রযুক্তি জমি নির্বাচন ও ভাল ফলন পে...,False


In [20]:
BN_TO_EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
def normalize(text):
    text = unicodedata.normalize("NFKC", str(text)).translate(BN_TO_EN).lower()
    text = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()
def tokens(text):
    return normalize(text).split()


In [21]:
# Use the same Exact Match and Token F1 definitions.
def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))
def token_f1(pred, gold):
    p, g = tokens(pred), tokens(gold)
    if not p or not g:
        return 0.0
    overlap = sum((Counter(p) & Counter(g)).values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(p), overlap / len(g)
    return 2 * precision * recall / (precision + recall)


In [22]:
# Use the same ROUGE-1 and ROUGE-2 definitions as RAG.
def rouge_n(pred, gold, n):
    p, g = tokens(pred), tokens(gold)
    if len(p) < n or len(g) < n:
        return 0.0
    pg = Counter(tuple(p[i:i+n]) for i in range(len(p)-n+1))
    gg = Counter(tuple(g[i:i+n]) for i in range(len(g)-n+1))
    overlap = sum((pg & gg).values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / sum(pg.values()), overlap / sum(gg.values())
    return 2 * precision * recall / (precision + recall)


In [23]:
# Use the same ROUGE-L definition as RAG.
def rouge_l(pred, gold):
    p, g = tokens(pred), tokens(gold)
    if not p or not g:
        return 0.0
    dp = [0] * (len(g) + 1)
    for x in p:
        new = [0]
        for j, y in enumerate(g, 1):
            new.append(dp[j-1] + 1 if x == y else max(dp[j], new[-1]))
        dp = new
    lcs = dp[-1]
    precision, recall = lcs / len(p), lcs / len(g)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


In [24]:
# Calculate all row-level metrics with the exact RAG definitions.
df = predictions_df.copy().fillna("")
df["Exact Match"] = [exact_match(p, g) for p, g in zip(df["prediction"], df["gold"])]
df["Fuzzy Match"] = [fuzz.token_set_ratio(normalize(p), normalize(g)) / 100 for p, g in zip(df["prediction"], df["gold"])]
df["Token F1"] = [token_f1(p, g) for p, g in zip(df["prediction"], df["gold"])]
df["ROUGE-1"] = [rouge_n(p, g, 1) for p, g in zip(df["prediction"], df["gold"])]
df["ROUGE-2"] = [rouge_n(p, g, 2) for p, g in zip(df["prediction"], df["gold"])]
df["ROUGE-L"] = [rouge_l(p, g) for p, g in zip(df["prediction"], df["gold"])]


In [25]:
# Calculate Corpus BLEU with the exact RAG SacreBLEU configuration.
bleu = BLEU(tokenize="none", smooth_method="exp", effective_order=True)
pred_texts = [" ".join(tokens(x)) for x in df["prediction"]]
gold_texts = [" ".join(tokens(x)) for x in df["gold"]]
corpus_bleu = bleu.corpus_score(pred_texts, [gold_texts]).score / 100
print("Corpus BLEU:", corpus_bleu)


Corpus BLEU: 0.06055973438495097


In [26]:
# Calculate multilingual BERTScore with the exact RAG model and options.
P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(), df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased", batch_size=4, device="cpu",
    idf=False, rescale_with_baseline=False, verbose=True,
)
df["BERT Precision"], df["BERT Recall"], df["BERT F1"] = P.cpu().numpy(), R.cpu().numpy(), F1.cpu().numpy()


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

calculating scores...
computing bert embedding.


  0%|          | 0/73 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/50 [00:00<?, ?it/s]

done in 19.06 seconds, 10.49 sentences/sec


In [27]:
# Build the final fine-tuned result table in the same order as RAG.
result = pd.DataFrame({
    "metric": ["Exact Match", "Fuzzy Match", "Corpus BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "Token F1", "BERT Precision", "BERT Recall", "BERT F1", "Truncated Outputs"],
    "score": [df["Exact Match"].mean(), df["Fuzzy Match"].mean(), corpus_bleu, df["ROUGE-1"].mean(), df["ROUGE-2"].mean(), df["ROUGE-L"].mean(), df["Token F1"].mean(), df["BERT Precision"].mean(), df["BERT Recall"].mean(), df["BERT F1"].mean(), df["truncated"].astype(str).str.lower().eq("true").sum()],
})
display(result)


,metric,score
0,Exact Match,0.000000
1,Fuzzy Match,0.544917
2,Corpus BLEU,0.060560
3,ROUGE-1,0.238008
4,ROUGE-2,0.096071
5,ROUGE-L,0.219441
6,Token F1,0.238008
7,BERT Precision,0.758354
8,BERT Recall,0.751644
9,BERT F1,0.754079


In [29]:
# Mount Google Drive and save the fine-tuned row-level predictions and final comparable result CSV.
from google.colab import drive
drive.mount('/content/drive')

RESULT_DIR = Path("/content/drive/MyDrive/finetuned_results")
RESULT_DIR.mkdir(exist_ok=True, parents=True)

df.to_csv(RESULT_DIR / "finetuned_predictions.csv", index=False, encoding="utf-8-sig")
result.to_csv(RESULT_DIR / "finetuned_result.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(trainer.state.log_history).to_csv(RESULT_DIR / "training_log.csv", index=False, encoding="utf-8-sig")
print("Saved to:", RESULT_DIR.resolve())

Mounted at /content/drive
Saved to: /content/drive/MyDrive/finetuned_results
